In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.sql.functions import to_timestamp

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

df_stage = spark.read.csv(f'/Volumes/comercial-dev/source/csvs/{csv_name}.csv', header=True, inferSchema=True)

In [0]:
dbutils.widgets.text("csv_name", "", "CSV name")
dbutils.widgets.text("stage_table_name", "", "Stage name")
dbutils.widgets.dropdown("ambiente", "dev", ["dev", "prod"])

csv_name = dbutils.widgets.get("csv_name")
stage_name = dbutils.widgets.get("stage_table_name")
ambiente = dbutils.widgets.get("ambiente")

In [0]:
logs_stage = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "stage",
    "catalog": f'comercial-{ambiente}'.upper(),
    "source_Name": csv_name.upper(),
    "target_Name": stage_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def create_or_replace_stage_table(stage):
    stage.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f'`comercial-{ambiente}`.`stage`.`{stage_name}`')

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f'`comercial-{ambiente}`.`corporate`.`logs`')
        
  

In [0]:


if df_stage:
    create_or_replace_stage_table(df_stage)

    logs_stage["status"] = "SUCCESS"
    logs_stage["count_Rows"] = df_stage.count()
    logs_stage["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs_stage])
    
    save_logs(df_logs)
else:
    print("CSV not exists")
